## Análise dos dados

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
import csv
import gc
import random

### Keystrokes

In [2]:
df_keystrokes = pd.read_csv("./dataset/keystrokes/24_keystrokes.txt", sep='\t')
print(df_keystrokes.columns)

Index(['PARTICIPANT_ID', 'TEST_SECTION_ID', 'SENTENCE', 'USER_INPUT',
       'KEYSTROKE_ID', 'PRESS_TIME', 'RELEASE_TIME', 'LETTER', 'KEYCODE'],
      dtype='str')


In [3]:
df_keystrokes.head()

,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,USER_INPUT,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,LETTER,KEYCODE
0,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5031,1471934627538,1471934627690,SHIFT,16
1,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5033,1471934627613,1471934627705,H,72
2,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5036,1471934627795,1471934627943,e,69
3,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5040,1471934627853,1471934627948,,32
4,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5045,1471934628057,1471934628162,m,77


In [4]:
df_keystrokes.tail()

,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,USER_INPUT,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,LETTER,KEYCODE
671,24,245,In a couple of weeks.,In a couple of weeks.,11459,1471934832485,1471934832581,e,69
672,24,245,In a couple of weeks.,In a couple of weeks.,11460,1471934832646,1471934832768,e,69
673,24,245,In a couple of weeks.,In a couple of weeks.,11461,1471934832793,1471934832871,k,75
674,24,245,In a couple of weeks.,In a couple of weeks.,11474,1471934832877,1471934832999,s,83
675,24,245,In a couple of weeks.,In a couple of weeks.,11477,1471934833085,1471934833171,.,190


In [5]:
print(df_keystrokes.shape)

(676, 9)


In [6]:
print("\n--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---")
# Transpose (.T) facilita a leitura no relatório
display(df_keystrokes.describe().T) 

print("\n--- ESTATÍSTICAS (Categóricas) ---")
display(df_keystrokes.describe(include=['O']).T)


--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---


,count,mean,std,min,25%,50%,75%,max
PARTICIPANT_ID,676.0,2.400000e+01,0.000000,2.400000e+01,2.400000e+01,2.400000e+01,2.400000e+01,2.400000e+01
TEST_SECTION_ID,676.0,1.762382e+02,38.190840,1.160000e+02,1.470000e+02,1.760000e+02,2.090000e+02,2.450000e+02
KEYSTROKE_ID,676.0,7.913348e+03,1900.143912,5.031000e+03,6.231500e+03,7.746500e+03,9.540250e+03,1.147700e+04
PRESS_TIME,676.0,1.471935e+12,60299.787440,1.471935e+12,1.471935e+12,1.471935e+12,1.471935e+12,1.471935e+12
RELEASE_TIME,676.0,1.471935e+12,60299.622147,1.471935e+12,1.471935e+12,1.471935e+12,1.471935e+12,1.471935e+12
KEYCODE,676.0,6.647189e+01,29.913120,8.000000e+00,6.500000e+01,7.200000e+01,8.000000e+01,2.220000e+02



--- ESTATÍSTICAS (Categóricas) ---


C:\Users\Diogo\AppData\Local\Temp\ipykernel_3724\2588143841.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_keystrokes.describe(include=['O']).T)


,count,unique,top,freq
SENTENCE,676,15,He disputed reports that Wheatley would be out...,86
USER_INPUT,676,15,He disputed reports that Wheatley would be out...,86
LETTER,676,45,,108


In [7]:
def process_keystroke(filepath):
    # 1. Ler o ficheiro
    df = pd.read_csv(
    filepath, 
    sep='\t', 
    encoding='latin-1',       # Ignora erros de caracteres estranhos do utf-8
    quoting=csv.QUOTE_NONE,   # Resolve o erro do "EOF inside string" (ignora aspas)
    on_bad_lines='skip',      # Resolve o erro do "saw 13 fields" (salta as linhas partidas)
    low_memory=False
    )
    
    # 2. Ordenar por sessão (TEST_SECTION_ID) e depois por tempo de pressão
    # Isto é vital para não calcularmos o "Flight Time" entre o fim de uma frase e o início de outra
    df = df.sort_values(by=['TEST_SECTION_ID', 'PRESS_TIME'])
    
    # Agrupar os dados por sessão para cálculos sequenciais seguros
    grouped = df.groupby('TEST_SECTION_ID')
    
    # --- MÉTRICAS BASE ---
    
    # Tempo de Retenção (Dwell Time): Release - Press
    df['DWELL_TIME'] = df['RELEASE_TIME'] - df['PRESS_TIME']
    
    # Tempo de Voo (Flight Time): Press(atual) - Release(anterior)
    df['PREV_RELEASE_TIME'] = grouped['RELEASE_TIME'].shift(1)
    df['FLIGHT_TIME'] = df['PRESS_TIME'] - df['PREV_RELEASE_TIME']
    
    # Tempo de Sobreposição (Overlap Time)
    # Se o Flight Time é negativo, significa que a tecla atual foi pressionada antes da anterior ser solta
    df['OVERLAP_TIME'] = df['FLIGHT_TIME'].apply(lambda x: abs(x) if x < 0 else 0)
    
    # --- LATÊNCIAS DE SEQUÊNCIAS ---
    
    # Latência de Dígrafos: Tempo entre pressões de 2 teclas consecutivas
    df['DIGRAPH_LATENCY'] = grouped['PRESS_TIME'].diff(1)
    
    # Latência de Trígrafos: Tempo entre pressões num intervalo de 3 teclas
    df['TRIGRAPH_LATENCY'] = grouped['PRESS_TIME'].diff(2)
    
    # --- FREQUÊNCIAS E PADRÕES DE TECLAS ESPECIAIS ---
    
    # Frequência de Correção: 8 (Backspace), 46 (Delete)
    df['IS_CORRECTION'] = df['KEYCODE'].isin([8, 46]).astype(int)
    
    # Teclas de Navegação: 35 (End), 36 (Home), 37 a 40 (Setas)
    df['IS_NAVIGATION'] = df['KEYCODE'].isin([35, 36, 37, 38, 39, 40]).astype(int)
    
    # Padrão Especial (ex: Ctrl + Backspace)
    # Identifica se a tecla anterior foi Ctrl (17) e a atual é Backspace (8)
    df['PREV_KEYCODE'] = grouped['KEYCODE'].shift(1)
    df['CTRL_BACKSPACE_PATTERN'] = ((df['KEYCODE'] == 8) & (df['PREV_KEYCODE'] == 17)).astype(int)
    
    # --- CADÊNCIA E VARIÂNCIA (Métricas de Janela Deslizante / Rolling) ---
    
    # Velocidade Global (WPM - Words Per Minute):
    # Assumimos que 1 palavra = 5 caracteres. Usamos uma janela das últimas 25 teclas.
    df['TIME_LAST_25_KEYS'] = grouped['PRESS_TIME'].diff(25) # tempo em milissegundos
    # Cálculo: (5 palavras) / (Tempo_em_minutos)
    df['ROLLING_WPM'] = 5 / (df['TIME_LAST_25_KEYS'] / 60000.0)
    
    # Variância Rítmica (Rhythm Variance):
    # Desvio padrão do Flight Time numa janela das últimas 10 teclas
    df['RHYTHM_VARIANCE'] = grouped['FLIGHT_TIME'].transform(lambda x: x.rolling(10, min_periods=2).std())
    
    # --- LIMPEZA E EXPORTAÇÃO ---
    
    features = [
        'PARTICIPANT_ID', 'TEST_SECTION_ID', 'PRESS_TIME', 'RELEASE_TIME',
        'DWELL_TIME', 'FLIGHT_TIME', 'OVERLAP_TIME', 
        'DIGRAPH_LATENCY', 'TRIGRAPH_LATENCY', 
        'IS_CORRECTION', 'IS_NAVIGATION', 'CTRL_BACKSPACE_PATTERN', 
        'ROLLING_WPM', 'RHYTHM_VARIANCE'
    ]

    # Preencher valores nulos gerados pelos "shifts" (inícios de frases) com 0
    df = df.fillna(0)
    
    return df

In [8]:
df_teclado = process_keystroke('./dataset/keystrokes/24_keystrokes.txt')
df_teclado.head(15)

,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,USER_INPUT,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,LETTER,KEYCODE,DWELL_TIME,...,OVERLAP_TIME,DIGRAPH_LATENCY,TRIGRAPH_LATENCY,IS_CORRECTION,IS_NAVIGATION,PREV_KEYCODE,CTRL_BACKSPACE_PATTERN,TIME_LAST_25_KEYS,ROLLING_WPM,RHYTHM_VARIANCE
0,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5031,1471934627538,1471934627690,SHIFT,16,152,...,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.000000
1,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5033,1471934627613,1471934627705,H,72,92,...,77.0,75.0,0.0,0,0,16.0,0,0.0,0.0,0.000000
2,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5036,1471934627795,1471934627943,e,69,148,...,0.0,182.0,257.0,0,0,72.0,0,0.0,0.0,118.086832
3,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5040,1471934627853,1471934627948,,32,95,...,90.0,58.0,240.0,0,0,69.0,0,0.0,0.0,100.380941
4,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5045,1471934628057,1471934628162,m,77,105,...,0.0,204.0,262.0,0,0,32.0,0,0.0,0.0,106.072302
5,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5052,1471934628089,1471934628298,a,65,209,...,73.0,32.0,236.0,0,0,77.0,0,0.0,0.0,98.745633
6,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5053,1471934628242,1471934628482,d,68,240,...,56.0,153.0,185.0,0,0,65.0,0,0.0,0.0,90.450907
7,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5055,1471934628375,1471934628543,e,69,168,...,107.0,133.0,286.0,0,0,68.0,0,0.0,0.0,89.423018
8,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5057,1471934628537,1471934628627,,32,90,...,6.0,162.0,295.0,0,0,69.0,0,0.0,0.0,83.192977
9,24,116,He made his mark by campaigning against hospit...,He made his mark by campaigning against hospit...,5061,1471934628757,1471934628830,h,72,73,...,0.0,220.0,382.0,0,0,32.0,0,0.0,0.0,93.640862


In [9]:
df_teclado.tail(15)

,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,USER_INPUT,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,LETTER,KEYCODE,DWELL_TIME,...,OVERLAP_TIME,DIGRAPH_LATENCY,TRIGRAPH_LATENCY,IS_CORRECTION,IS_NAVIGATION,PREV_KEYCODE,CTRL_BACKSPACE_PATTERN,TIME_LAST_25_KEYS,ROLLING_WPM,RHYTHM_VARIANCE
661,24,245,In a couple of weeks.,In a couple of weeks.,11333,1471934829473,1471934829674,p,80,201,...,0.0,266.0,559.0,0,0,85.0,0,0.0,0.000000,291.654762
662,24,245,In a couple of weeks.,In a couple of weeks.,11390,1471934829555,1471934829718,l,76,163,...,119.0,82.0,348.0,0,0,80.0,0,0.0,0.000000,295.984703
663,24,245,In a couple of weeks.,In a couple of weeks.,11395,1471934829818,1471934829920,e,69,102,...,0.0,263.0,345.0,0,0,76.0,0,0.0,0.000000,295.791143
664,24,245,In a couple of weeks.,In a couple of weeks.,11398,1471934830833,1471934830933,o,79,100,...,0.0,1015.0,1278.0,0,0,69.0,0,0.0,0.000000,383.685766
665,24,245,In a couple of weeks.,In a couple of weeks.,11403,1471934831193,1471934831251,BKSP,8,58,...,0.0,360.0,1375.0,1,0,79.0,0,0.0,0.000000,370.682868
666,24,245,In a couple of weeks.,In a couple of weeks.,11407,1471934831566,1471934831675,,32,109,...,0.0,373.0,733.0,0,0,8.0,0,0.0,0.000000,370.612241
667,24,245,In a couple of weeks.,In a couple of weeks.,11410,1471934831800,1471934831893,o,79,93,...,0.0,234.0,607.0,0,0,32.0,0,0.0,0.000000,358.619204
668,24,245,In a couple of weeks.,In a couple of weeks.,11454,1471934831887,1471934832003,f,70,116,...,6.0,87.0,321.0,0,0,79.0,0,0.0,0.000000,349.311847
669,24,245,In a couple of weeks.,In a couple of weeks.,11456,1471934832103,1471934832183,,32,80,...,0.0,216.0,303.0,0,0,70.0,0,0.0,0.000000,277.871593
670,24,245,In a couple of weeks.,In a couple of weeks.,11458,1471934832278,1471934832526,w,87,248,...,0.0,175.0,391.0,0,0,32.0,0,0.0,0.000000,280.139668


In [15]:
def agrupar_todos_keystrokes(path, caminho_guardar):
    padrao_pesquisa = os.path.join(path, '*_keystrokes.txt')
    lista_ficheiros = glob.glob(padrao_pesquisa)
    
    if not lista_ficheiros:
        print(f"Nenhum ficheiro encontrado na pasta: {path}")
        return False
    
    print(f"Encontrados {len(lista_ficheiros)} ficheiros. A iniciar processamento...")
    
    # 1. Baralhar a lista para não haver enviesamentos
    random.shuffle(lista_ficheiros)

    # 2. Cortar a lista para 1/4 do tamanho
    um_quarto = len(lista_ficheiros) // 4
    lista_ficheiros = lista_ficheiros[:um_quarto]
    
    print(f"A usar apenas 1/4 dos dados. Ficheiros a processar: {len(lista_ficheiros)}")
    
    lista_dataframes = []
    primeiro_lote = True

    # 3. Iterar sobre cada ficheiro
    for ficheiro in tqdm(lista_ficheiros, desc="A processar utilizadores"):
        try:
            # Processar o ficheiro
            df_utilizador = process_keystroke(ficheiro)
            
            # GARANTIA DE PRIVACIDADE E PREVENÇÃO DE ERROS:
            # Se a coluna LETTER ou USER_INPUT ainda existirem, removemos!
            colunas_a_remover = ['LETTER', 'USER_INPUT']
            df_utilizador = df_utilizador.drop(columns=[col for col in colunas_a_remover if col in df_utilizador.columns], errors='ignore')

            # Adicionar à lista
            lista_dataframes.append(df_utilizador)
            
        except Exception as e:
            pass # Ignoramos as linhas corrompidas

        # O SEGREDO: Escrever no disco a cada 5000 utilizadores
        if len(lista_dataframes) >= 5000:
            df_chunk = pd.concat(lista_dataframes, ignore_index=True)
            df_chunk.to_csv(caminho_guardar, mode='a', header=primeiro_lote, index=False)
            
            primeiro_lote = False 
            lista_dataframes = [] 
            gc.collect()          

    # Gravar os últimos utilizadores que sobraram
    if len(lista_dataframes) > 0:
        df_chunk = pd.concat(lista_dataframes, ignore_index=True)
        df_chunk.to_csv(caminho_guardar, mode='a', header=primeiro_lote, index=False)
            
    print(f"\nTodos os lotes foram gravados com sucesso em: {caminho_guardar}")
    return True # Já não devolvemos o DataFrame parcial, apenas confirmamos o sucesso

In [16]:
pasta_keystrokes = 'dataset/keystrokes/'
caminho_guardar = 'dataset/keystroke_dataset_processado.csv'

# MUITO IMPORTANTE: Se o ficheiro já existir de uma execução anterior falhada, apaga-o.
# Senão o 'mode="a"' vai adicionar dados ao ficheiro velho!
if os.path.exists(caminho_guardar):
    os.remove(caminho_guardar)

# Executar a função
sucesso = agrupar_todos_keystrokes(pasta_keystrokes, caminho_guardar)

if sucesso:
    print("\nO dataset completo (CSV) está pronto!")

Encontrados 168593 ficheiros. A iniciar processamento...
A usar apenas 1/4 dos dados. Ficheiros a processar: 42148


A processar utilizadores: 100%|██████████| 42148/42148 [19:05<00:00, 36.79it/s]   



Todos os lotes foram gravados com sucesso em: dataset/keystroke_dataset_processado.csv

O dataset completo (CSV) está pronto!


In [17]:
df_test = pd.read_csv("./dataset/keystroke_dataset_processado.csv")
df_test.head()

,PARTICIPANT_ID,TEST_SECTION_ID,SENTENCE,KEYSTROKE_ID,PRESS_TIME,RELEASE_TIME,KEYCODE,DWELL_TIME,PREV_RELEASE_TIME,FLIGHT_TIME,OVERLAP_TIME,DIGRAPH_LATENCY,TRIGRAPH_LATENCY,IS_CORRECTION,IS_NAVIGATION,PREV_KEYCODE,CTRL_BACKSPACE_PATTERN,TIME_LAST_25_KEYS,ROLLING_WPM,RHYTHM_VARIANCE
0,209489.0,2269959,Thank you for your support.,107828239.0,1.474661e+12,1.474661e+12,16.0,241.0,0.000000e+00,0.0,0.0,0.0,0.0,0,0,0.0,0,0.0,0.0,0.000000
1,209489.0,2269959,Thank you for your support.,107828242.0,1.474661e+12,1.474661e+12,84.0,102.0,1.474661e+12,-64.0,64.0,177.0,0.0,0,0,16.0,0,0.0,0.0,0.000000
2,209489.0,2269959,Thank you for your support.,107828245.0,1.474661e+12,1.474661e+12,72.0,98.0,1.474661e+12,40.0,0.0,142.0,319.0,0,0,84.0,0,0.0,0.0,73.539105
3,209489.0,2269959,Thank you for your support.,107828249.0,1.474661e+12,1.474661e+12,65.0,113.0,1.474661e+12,4.0,0.0,102.0,244.0,0,0,72.0,0,0.0,0.0,52.814140
4,209489.0,2269959,Thank you for your support.,107828252.0,1.474661e+12,1.474661e+12,78.0,108.0,1.474661e+12,-17.0,17.0,96.0,198.0,0,0,65.0,0,0.0,0.0,43.430980


In [19]:
print(df_test.shape)

(30774437, 20)


### Mouse

In [20]:
df_mouse = pd.read_csv("dataset/Mouse-Dynamics/training_files/user7/session_1060325796")
print(df_mouse.columns)

Index(['record timestamp', 'client timestamp', 'button', 'state', 'x', 'y'], dtype='str')


In [21]:
df_mouse.head()

,record timestamp,client timestamp,button,state,x,y
0,0.00,0.000,NoButton,Move,374,232
1,0.16,0.016,NoButton,Move,380,240
2,0.16,0.016,NoButton,Move,388,247
3,0.16,0.047,NoButton,Move,392,248
4,0.16,0.172,NoButton,Move,393,248


In [22]:
df_mouse.tail()

,record timestamp,client timestamp,button,state,x,y
43479,2452.280,2452.196,NoButton,Move,1182,241
43480,2452.280,2452.211,NoButton,Move,1182,240
43481,2452.280,2452.227,NoButton,Move,1182,239
43482,2452.280,2452.289,Left,Pressed,1182,239
43483,2452.392,2452.399,Left,Released,1182,239


In [23]:
print(df_mouse.shape)

(43484, 6)


In [24]:
print("\n--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---")
# Transpose (.T) facilita a leitura no relatório
display(df_mouse.describe().T) 

print("\n--- ESTATÍSTICAS (Categóricas) ---")
display(df_mouse.describe(include=['O']).T)


--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---


,count,mean,std,min,25%,50%,75%,max
record timestamp,43484.0,1282.979799,688.934533,0.0,636.111,1339.499,1857.068,2452.392
client timestamp,43484.0,1282.919711,688.937502,0.0,636.044,1339.417,1857.040,2452.399
x,43484.0,378.596748,232.144165,0.0,240.000,332.000,453.000,1223.000
y,43484.0,383.183470,184.892827,0.0,249.750,373.000,518.250,799.000



--- ESTATÍSTICAS (Categóricas) ---


C:\Users\Diogo\AppData\Local\Temp\ipykernel_3724\935740185.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_mouse.describe(include=['O']).T)


,count,unique,top,freq
button,43484,4,NoButton,41414
state,43484,6,Move,39599


In [25]:
def process_mouse_file(filepath, file_label_is_illegal=None):
    # 1. Ler o ficheiro. Segundo o teu exemplo, as colunas são:
    # record timestamp, client timestamp, button, state, x, y
    df = pd.read_csv(filepath, sep=',')
    
    # Garantir nomes consistentes e remover espaços nos cabeçalhos, caso existam
    df.columns = df.columns.str.strip()
    
    # Ordenar cronologicamente
    df = df.sort_values(by='client timestamp')
    
    # 2. Calcular Derivadas Base (Tempo e Distância)
    # A diferença de tempo (dt) em segundos
    df['dt'] = df['client timestamp'].diff().fillna(0)
    
    # Evitar divisão por zero (Adicionamos um epsilon, ou consideramos dt = 0.001 se for muito pequeno)
    df['dt'] = np.where(df['dt'] < 0.001, 0.001, df['dt'])

    # Diferença espacial (dx, dy) e distância euclidiana percorrida
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2)
    
    # --- 1. Velocidade e Aceleração ---
    df['velocity'] = df['distance'] / df['dt']
    df['acceleration'] = df['velocity'].diff().fillna(0) / df['dt']
    
    # --- 2. Curvatura e Ângulo de Movimento ---
    # Calcular o ângulo de movimento (em radianos) usando arctan2
    df['angle'] = np.arctan2(df['dy'], df['dx'])
    
    # Curvatura (Jitter) aproximada como a taxa de variação do ângulo sobre a distância
    # Quanto mais o ângulo muda a cada movimento, maior a curvatura.
    df['angle_change'] = df['angle'].diff().abs().fillna(0)
    df['curvature'] = df['angle_change'] / (df['distance'] + 1e-6)
    
    # --- 3. Segmentação por Ações (Strokes) ---
    # Um "stroke" de rato começa quando o rato se move e termina quando ele para.
    # Vamos definir um Idle Time (Pausa) de 200ms (0.2s) para dividir os movimentos
    df['is_idle'] = (df['dt'] > 0.2).astype(int)
    # Criar um ID único para cada "Action Stroke" contínua
    df['stroke_id'] = df['is_idle'].cumsum()
    
    # --- 4. Distribuição Direcional ---
    # Classificar o movimento em 8 direções baseadas no ângulo
    df['direction_bin'] = np.floor((df['angle'] + np.pi) / (np.pi / 4)) % 8
    
    # --- 5. Eventos de Botões (Click e Drag) ---
    # Filtrar apenas os cliques (esquerdos) para analisar retenção
    # Como o Balabit regista "Pressed" e "Released", podemos medir o Dwell Time do clique
    df['is_pressed'] = (df['state'] == 'Pressed').astype(int)
    df['is_released'] = (df['state'] == 'Released').astype(int)
    df['is_drag'] = (df['state'] == 'Drag').astype(int)
    
    # Para o projeto base, guardamos as labels do estado. O cálculo exato do Dwell  de um clique exige separar os Pressed/Released de cada stroke, o que fazemos  indiretamente analisando o estado ao longo do tempo na LSTM).

    # --- 6. Ultrapassagem do Alvo (Overshoot) ---
    # O overshoot acontece quando tu vais depressa numa direção e, logo a seguir (num dt pequeno), mudas drasticamente de direção (ângulo > 150 graus ou ~2.6 radianos) e a velocidade cai muito.
    df['is_overshoot'] = ((df['angle_change'] > 2.6) & (df['velocity'].shift(1) > df['velocity'])).astype(int)

    # Limpeza de infinitos (caso a divisão por zero tenha gerado infs)
    df = df.replace([np.inf, -np.inf], 0)
    
    # --- 7. Seleção de Features ---
    features = [
        'client timestamp', 'dt', 'dx', 'dy', 'distance', 
        'velocity', 'acceleration', 'angle', 'angle_change', 'curvature',
        'is_idle', 'stroke_id', 'direction_bin', 
        'is_pressed', 'is_released', 'is_drag', 'is_overshoot'
    ]

    df_features = df[features].copy()
    
    # Se passaste a tag (is_illegal) lida do ficheiro public_labels, nós guardamo-la!
    if file_label_is_illegal is not None:
        df_features['is_illegal'] = file_label_is_illegal
        
    return df_features

In [26]:
df_rato = process_mouse_file('dataset/Mouse-Dynamics/training_files/user7/session_1060325796')
df_rato.head(10)

,client timestamp,dt,dx,dy,distance,velocity,acceleration,angle,angle_change,curvature,is_idle,stroke_id,direction_bin,is_pressed,is_released,is_drag,is_overshoot
0,0.000,0.001,0.0,0.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0,0,4.0,0,0,0,0
1,0.016,0.016,6.0,8.0,10.000000,624.999989,3.906250e+04,0.927295,0.927295,0.092730,0,0,5.0,0,0,0,0
2,0.016,0.001,8.0,7.0,10.630146,10630.145813,1.000515e+07,0.718830,0.208465,0.019611,0,0,4.0,0,0,0,0
3,0.047,0.031,4.0,1.0,4.123106,133.003407,-3.386175e+05,0.244979,0.473851,0.114926,0,0,4.0,0,0,0,0
4,0.172,0.125,1.0,0.0,1.000000,8.000000,-1.000027e+03,0.000000,0.244979,0.244978,0,0,4.0,0,0,0,0
5,0.188,0.016,5.0,6.0,7.810250,488.140610,3.000879e+04,0.876058,0.876058,0.112168,0,0,5.0,0,0,0,0
6,0.203,0.015,3.0,0.0,3.000000,199.999998,-1.920937e+04,0.000000,0.876058,0.292019,0,0,4.0,0,0,0,0
7,0.219,0.016,0.0,-5.0,5.000000,312.500003,7.031250e+03,-1.570796,1.570796,0.314159,0,0,2.0,0,0,0,0
8,0.234,0.015,0.0,-2.0,2.000000,133.333332,-1.194444e+04,-1.570796,0.000000,0.000000,0,0,2.0,0,0,0,0
9,0.250,0.016,4.0,0.0,4.000000,250.000003,7.291667e+03,0.000000,1.570796,0.392699,0,0,4.0,0,0,0,0


In [29]:
def construir_dataset_rato_global(diretorio_base):
    # 1. Ler o ficheiro de labels (public_labels.txt)
    caminho_labels = os.path.join(diretorio_base, 'public_labels.csv')
    
    print("A ler o mapeamento de labels...")
    # Assumimos separação por vírgula. Ajusta o 'sep' se for por tabulação ('\t')
    df_labels = pd.read_csv(caminho_labels, sep=',') 
    
    # Criar um dicionário para procura ultra-rápida. 
    # Exemplo: {'session_0061629194': 1, 'session_1060325796': 0}
    dicionario_labels = {}
    for index, row in df_labels.iterrows():
        # A primeira coluna costuma ser o filename. Extraímos apenas o nome da sessão final.
        nome_ficheiro = str(row.iloc[0]).split('/')[-1].replace('.csv', '') 
        label = int(row.iloc[1]) # 0 (Dono) ou 1 (Ilegal)
        dicionario_labels[nome_ficheiro] = label

    # 2. Procurar recursivamente por todas as sessões dentro da pasta Mouse-Dynamics
    # Isto vai procurar dentro de test_files/userX/ e training_files/userX/
    padrao_pesquisa = os.path.join(diretorio_base, '**', 'session_*')
    lista_ficheiros = glob.glob(padrao_pesquisa, recursive=True)
    
    # Filtrar apenas ficheiros (para o caso de haver pastas com nome parecido)
    lista_ficheiros = [f for f in lista_ficheiros if os.path.isfile(f)]
    
    if not lista_ficheiros:
        print("Erro: Nenhum ficheiro encontrado. Verifica o caminho da pasta.")
        return None
        
    print(f"Encontradas {len(lista_ficheiros)} sessões de rato. A iniciar extração cinemática...")
    
    lista_dataframes = []
    
    # 3. Iterar e extrair as features matemáticas com a barra de progresso
    for ficheiro in tqdm(lista_ficheiros, desc="A processar Rato"):
        try:
            # Extrair a sessão e o ID do utilizador do próprio caminho do ficheiro
            # Como estás em Windows, usamos os.sep para dividir o caminho ('\')
            partes_caminho = ficheiro.split(os.sep)
            nome_sessao = partes_caminho[-1]                  # Ex: 'session_0061629194'
            user_id = partes_caminho[-2].replace('user', '')  # Transforma 'user7' em '7'
            
            # 4. Ir buscar a label ao nosso dicionário
            # Se não encontrar a sessão no txt (dados incompletos), ignora e salta
            if nome_sessao not in dicionario_labels:
                continue 
                
            is_illegal_label = dicionario_labels[nome_sessao]
            
            # 5. Processar o ficheiro com a nossa função avançada
            df_sessao = process_mouse_file(ficheiro, file_label_is_illegal=is_illegal_label)
            
            # Adicionar as colunas de identificação no início do DataFrame
            df_sessao.insert(0, 'USER_ID', user_id)
            df_sessao.insert(1, 'SESSION_ID', nome_sessao)
            
            lista_dataframes.append(df_sessao)
            
        except Exception as e:
            # Tal como no teclado, se um ficheiro estiver corrompido, salta à frente.
            pass

    # 6. Fundir tudo num único ecrã
    print("\nA fundir todos os dados cinemáticos num único DataFrame mestre...")
    df_final = pd.concat(lista_dataframes, ignore_index=True)
    
    return df_final

In [30]:
# Define a pasta raiz do rato (onde estão as pastas test_files e training_files)
pasta_rato = 'dataset/Mouse-Dynamics/'

df_global_rato = construir_dataset_rato_global(pasta_rato)

if df_global_rato is not None:
    print(f"\nProcessamento de rato concluído!")
    print(f"Total de registos (linhas): {len(df_global_rato)}")
    print(f"Total de sessões processadas: {df_global_rato['SESSION_ID'].nunique()}")
    print(f"Distribuição Legaís (0) vs Ilegais (1):\n{df_global_rato['is_illegal'].value_counts()}")
    
    # Guardar em formato de alta performance (fortemente recomendado)
    caminho_guardar = 'dataset/mouse_dataset_processado.parquet'
    print(f"A guardar ficheiro final em: {caminho_guardar}")
    df_global_rato.to_parquet(caminho_guardar, index=False)
    
    # Se ainda não tiveres o pyarrow instalado, usa CSV (vai demorar mais a gravar)
    # df_global_rato.to_csv('dataset/mouse_dataset_processado.csv', index=False)
    print("Ficheiro guardado e pronto para a Inteligência Artificial!")

A ler o mapeamento de labels...
Encontradas 1675 sessões de rato. A iniciar extração cinemática...


A processar Rato: 100%|██████████| 1675/1675 [00:13<00:00, 122.15it/s]



A fundir todos os dados cinemáticos num único DataFrame mestre...

Processamento de rato concluído!
Total de registos (linhas): 1157823
Total de sessões processadas: 816
Distribuição Legaís (0) vs Ilegais (1):
is_illegal
0    723358
1    434465
Name: count, dtype: int64
A guardar ficheiro final em: dataset/mouse_dataset_processado.parquet
Ficheiro guardado e pronto para a Inteligência Artificial!


In [32]:
df_test_mouse = pd.read_parquet("./dataset/mouse_dataset_processado.parquet")
df_test_mouse.head()

,USER_ID,SESSION_ID,client timestamp,dt,dx,dy,distance,velocity,acceleration,angle,angle_change,curvature,is_idle,stroke_id,direction_bin,is_pressed,is_released,is_drag,is_overshoot,is_illegal
0,12,session_0126772600,0.000,0.001,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,4.0,0,0,0,0,1
1,12,session_0126772600,0.000,0.001,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,4.0,1,0,0,0,1
2,12,session_0126772600,0.031,0.031,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,4.0,0,1,0,0,1
3,12,session_0126772600,0.359,0.328,3.0,-2.0,3.605551,10.992534,33.513824,-0.588003,0.588003,0.163083,1,1,3.0,0,0,0,0,1
4,12,session_0126772600,0.452,0.093,39.0,-33.0,51.088159,549.335045,5788.629152,-0.702257,0.114254,0.002236,0,1,3.0,0,0,0,0,1


In [33]:
print("\n--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---")
# Transpose (.T) facilita a leitura no relatório
display(df_test_mouse.describe().T) 

print("\n--- ESTATÍSTICAS (Categóricas) ---")
display(df_test_mouse.describe(include=['O']).T)


--- ESTATÍSTICAS DESCRITIVAS (Numéricas) ---


,count,mean,std,min,25%,50%,75%,max
client timestamp,1157823.0,2.770784e+02,4.192602e+02,0.000000e+00,68.625000,165.236000,330.239000,7.317367e+03
dt,1157823.0,3.688122e-01,1.470190e+01,1.000000e-03,0.016000,0.109000,0.125000,6.743378e+03
dx,1157823.0,-4.464327e-02,8.148207e+02,-6.529900e+04,-3.000000,0.000000,3.000000,6.547600e+04
dy,1157823.0,-6.425939e-02,8.186751e+02,-6.552400e+04,-3.000000,0.000000,2.000000,6.553500e+04
distance,1157823.0,5.531945e+01,1.153734e+03,0.000000e+00,1.414214,6.324555,25.000000,9.263878e+04
velocity,1157823.0,1.176915e+04,6.191962e+05,0.000000e+00,21.276596,139.754249,647.640931,9.231778e+07
acceleration,1157823.0,9.250865e+06,6.231361e+08,-9.090525e+10,-2848.920698,0.000000,2478.290099,9.231661e+10
angle,1157823.0,9.721709e-02,1.737319e+00,-3.140761e+00,-1.438245,0.000000,1.570796,3.141593e+00
angle_change,1157823.0,8.508143e-01,1.395749e+00,0.000000e+00,0.033100,0.244979,0.850656,6.280504e+00
curvature,1157823.0,7.626436e+04,4.033824e+05,0.000000e+00,0.000695,0.013355,0.149634,3.141593e+06



--- ESTATÍSTICAS (Categóricas) ---


C:\Users\Diogo\AppData\Local\Temp\ipykernel_3724\1384124422.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df_test_mouse.describe(include=['O']).T)


,count,unique,top,freq
USER_ID,1157823,10,7,169335
SESSION_ID,1157823,816,session_3659572440,12672
